In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q pymupdf
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 60.0 MB/s eta 0:00:00


In [ ]:
import fitz
import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [2]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 68.3 MB/s eta 0:00:00


In [1]:
import fitz

print("PyMuPDF imported successfully!")

PyMuPDF imported successfully!


In [3]:
# ===========================================
# Read WHO COVID Guidelines PDF
# ===========================================

import fitz

pdf_path = "/content/WHO-COVID GUIDELINES.pdf"

doc = fitz.open(pdf_path)

text = ""

for page in doc:
    text += page.get_text()

print("PDF Loaded Successfully!")
print("Number of Pages:", len(doc))
print("Total Characters:", len(text))

print("\nFirst 1000 Characters:\n")
print(text[:1000])

PDF Loaded Successfully!
Number of Pages: 58
Total Characters: 165730

First 1000 Characters:

 
1 
 
 
 
 
 
 
 
 
Manual for respiratory virus vaccination coverage  
Monitoring and reporting seasonal 
influenza, COVID-19, and respiratory 
syncytial virus immunization  
 
 
 
 
 
Manual for respiratory virus vaccination coverage  
Monitoring and reporting seasonal 
influenza, COVID-19, and respiratory 
syncytial virus immunization  
 
 
 
 
Manual for respiratory virus vaccination coverage: monitoring and reporting seasonal influenza, COVID-19, and 
respiratory syncytial virus immunization 
ISBN 978-92-4-011882-9 (electronic version) 
ISBN 978-92-4-011883-6 (print version) 
© World Health Organization 2026 
Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 
3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo).  
Under the terms of this licence, you may copy, redistribute and adapt the wo

In [4]:
# ===========================================
# Split Text into Chunks
# ===========================================

chunk_size = 500
overlap = 100

chunks = []

start = 0

while start < len(text):

    end = start + chunk_size

    chunks.append(text[start:end])

    start += chunk_size - overlap

print("Number of Chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0])

Number of Chunks: 415

First Chunk:

 
1 
 
 
 
 
 
 
 
 
Manual for respiratory virus vaccination coverage  
Monitoring and reporting seasonal 
influenza, COVID-19, and respiratory 
syncytial virus immunization  
 
 
 
 
 
Manual for respiratory virus vaccination coverage  
Monitoring and reporting seasonal 
influenza, COVID-19, and respiratory 
syncytial virus immunization  
 
 
 
 
Manual for respiratory virus vaccination coverage: monitoring and reporting seasonal influenza, COVID-19, and 
respiratory syncytial virus immunizati


In [5]:
# ===========================================
# Generate Embeddings
# ===========================================

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding Shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embedding Shape: (415, 384)


In [6]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 90.0 MB/s eta 0:00:00


In [1]:
import faiss

print("FAISS Imported Successfully!")

FAISS Imported Successfully!


In [6]:
# ===========================================
# Build FAISS Index
# ===========================================

import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS Index Created!")
print("Total Chunks Indexed:", index.ntotal)

FAISS Index Created!
Total Chunks Indexed: 415


In [7]:
# ===========================================
# Retrieve Relevant Chunks
# ===========================================

def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, top_k)

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

print("Retriever Ready!")

Retriever Ready!


In [8]:
# ===========================================
# Test Retrieval
# ===========================================

query = "What are the symptoms of COVID-19?"

results = retrieve(query)

for i, chunk in enumerate(results):

    print("="*60)
    print(f"Chunk {i+1}")
    print("="*60)
    print(chunk[:600])
    print()

Chunk 1
te]. World Health Organization; 2026 
(https://www.who.int/publications/m/item/covid-19-global-risk-assessment). Licence: CC BY-
NC-SA 3.0 IGO. 
3. 
Chen C, Haupert S, Zimmermann L, Shi X, Fritsche L, Mukherjee B. Global prevalence of 
post-coronavirus disease 2019 (COVID-19) condition or long COVID: a meta-analysis and 
systematic 
review. 
The 
Journal 
of 
Infectious 
Diseases. 
2022;226(9):1593-607 
(https://doi.org/10.1093/infdis/jiac136). 
4. 
Li Y WX, Blau DM, Caballero MT, Feikin DR, Gil

Chunk 2
year (1). SARS-CoV-2 continues to circulate widely, with ongoing impacts from acute infections 
and the growing burden of post-COVID-19 condition (2, 3). RSV causes an estimated 3.6 million 
hospitalizations and over 100 000 deaths in children aged under five years annually (4). Although 
the global estimates for RSV adult hospitalizations and deaths are lacking, the burden, 
particularly in older adults, is expected to be high (5). 
Seasonal influenza, COVID-19, and RSV immuni

In [10]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("Generator Loaded Successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

Generator Loaded Successfully!


In [13]:
def healthcare_assistant(question):

    retrieved = retrieve(question)

    context = "\n\n".join(retrieved)

    prompt = f"""
Answer ONLY using the information in the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    answer = result[0]["generated_text"]

    print("="*70)
    print("Question:")
    print(question)

    print("\nAnswer:")
    print(answer)

    print("\nConfidence: High (Retrieved from WHO Guidelines)")

    print("\nRetrieved Sources:")

    for i, chunk in enumerate(retrieved):
        print(f"\nSource {i+1}:")
        print(chunk[:250], "...")

In [14]:
healthcare_assistant("How does COVID-19 spread?")

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
How does COVID-19 spread?

Answer:

Answer ONLY using the information in the context below.

Context:
te]. World Health Organization; 2026 
(https://www.who.int/publications/m/item/covid-19-global-risk-assessment). Licence: CC BY-
NC-SA 3.0 IGO. 
3. 
Chen C, Haupert S, Zimmermann L, Shi X, Fritsche L, Mukherjee B. Global prevalence of 
post-coronavirus disease 2019 (COVID-19) condition or long COVID: a meta-analysis and 
systematic 
review. 
The 
Journal 
of 
Infectious 
Diseases. 
2022;226(9):1593-607 
(https://doi.org/10.1093/infdis/jiac136). 
4. 
Li Y WX, Blau DM, Caballero MT, Feikin DR, Gil

int/server/api/core/bitstreams/4f8b7520-1d1c-471d-b584-
8dd16cb79bed/content). Licence: CC BY-NC-SA 3.0 IGO. 
8. 
WHO SAGE roadmap on uses of COVID-19 vaccines in the context of Omicron and 
substantial population immunity: an approach to optimize the global impact of COVID-19 
vaccines. Version 10 November 2023. Geneva: World Health Organization; 2023 
(https://iris.who.int/server/a

In [15]:
import gradio as gr

def chatbot(question):

    retrieved = retrieve(question)

    context = "\n\n".join(retrieved)

    prompt = f"""
Answer ONLY using the information below.

Context:
{context}

Question:
{question}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    return result[0]["generated_text"]

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(label="Ask a Healthcare Question"),
    outputs=gr.Textbox(label="Answer"),
    title="Healthcare Evidence-Based Assistant",
    description="Answers are generated from WHO COVID Guidelines using Retrieval-Augmented Generation (RAG)."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88ac6f74fb733283d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
